In [2]:
import os 
from pathlib import Path
import pandas as pd 


In [3]:
# os.chdir(r"C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte")
PROJECT_DIR = Path(r"C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte")
RAW_DATA = PROJECT_DIR / "data" / "raw" / "Bretagne"


In [4]:
all_data = []

for data in RAW_DATA.iterdir():
    df = pd.read_csv(
        data,
        sep="\t",
        encoding="latin1",
        dtype=str,
        na_values=["-", "", "NA", "N/A" ,"ND"],
        keep_default_na=True,
        index_col= False
    )

    df = df.iloc[:-1].copy()

    df = df.loc[:, ~df.columns.str.startswith("Unnamed")].copy()

    df["datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Heures"], errors="coerce", dayfirst=True
    )

    df = df[df["datetime"].dt.minute.isin([0, 30])].copy()

    all_data.append(df)


C:\Users\user\AppData\Local\Temp\ipykernel_9744\1244513763.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


In [5]:
reference_columns = set(all_data[0].columns)

for i, data in enumerate(all_data):
    current_columns = set(data.columns)

    missing = reference_columns - current_columns
    extra = current_columns - reference_columns

    print(f"\nDataFrame {all_data[i]['Date'].str[:4].iloc[0]}")

    if not missing and not extra:
        print("✅ Même variables")
    else:
        print("❌ Variables différentes")
        print("Colonnes manquantes :", missing)
        print("Colonnes supplémentaires :", extra)



DataFrame 2013
✅ Même variables

DataFrame 2014
✅ Même variables

DataFrame 2015
✅ Même variables

DataFrame 2016
✅ Même variables

DataFrame 2017
✅ Même variables

DataFrame 2018
✅ Même variables

DataFrame 2019
✅ Même variables

DataFrame 2020
❌ Variables différentes
Colonnes manquantes : set()
Colonnes supplémentaires : {'Flux physiques de Bretagne vers Auvergne-Rhône-Alpes', 'Flux physiques Belgique vers Bretagne', 'Flux physiques de PACA vers Bretagne', 'Flux physiques de Bretagne vers Occitanie', 'Flux physiques de Bretagne vers PACA', 'TCO Bioénergies (%)', 'Flux physiques de Normandie vers Bretagne', 'Flux physiques Royaume-Uni vers Bretagne', 'Flux physiques Espagne vers Bretagne', 'Flux physiques de Bretagne vers Ile-de-France', 'Flux physiques Luxembourg vers Bretagne', 'TCH Eolien (%)', 'Flux physiques de Bretagne vers Allemagne', 'TCO Nucléaire (%)', 'Flux physiques Italie vers Bretagne', "Flux physiques d'Ile-de-France vers Bretagne", 'TCH Nucléaire (%)', 'Flux physiques

In [6]:
os.chdir(PROJECT_DIR)
# os.getcwd()

In [7]:
# # all_data[6] # 6
# df = pd.read_csv(r"data\raw\Bretagne\eCO2mix_RTE_Bretagne_Annuel-Definitif_2019.xls" ,
#             sep ="\t" ,
#             encoding= "latin1",
#             index_col =False)

# print(df.index[:5])
# print(df.columns.tolist())


In [8]:
df = pd.concat(all_data , ignore_index =False , sort = False)

In [9]:
pd.set_option("Display.max_rows",99)
pd.set_option("Display.max_columns", 99)


In [10]:
(df.isna().sum()).sort_values(ascending = False) / len(df)

Flux physiques de Grand-Est vers Bretagne                  1.000000
Flux physiques de Hauts-de-France vers Bretagne            1.000000
Flux physiques de Centre-Val de Loire vers Bretagne        1.000000
Flux physiques de Bretagne vers Bretagne                   1.000000
Flux physiques de Bretagne vers Suisse                     1.000000
Flux physiques de Bretagne vers Luxembourg                 1.000000
TCH Nucléaire (%)                                          1.000000
Flux physiques de Bretagne vers Royaume-Uni                1.000000
Flux physiques d'Auvergne-Rhône-Alpes vers Bretagne        1.000000
Flux physiques de Bourgogne-Franche-Comté vers Bretagne    1.000000
Flux physiques de Bretagne vers Nouvelle-Aquitaine         1.000000
Flux physiques de Bretagne vers Ile-de-France              1.000000
Flux physiques de Bretagne vers Bretagne.1                 1.000000
Flux physiques de Bretagne vers Centre-Val de Loire        1.000000
Flux physiques de Bretagne vers Auvergne-Rhône-A

In [11]:
df = df.dropna(axis=1 , how="all")#.isna().sum().sort_values(ascending=False) / len(df)

In [12]:
ENERGY_COLUMNS = [
    "Consommation",
    "Thermique",
    "Nucléaire",
    "Eolien",
    "Solaire",
    "Hydraulique",
    "Pompage",
    "Bioénergies",
    "Ech. physiques",
    "Stockage batterie",
    "Déstockage batterie",
    "Eolien terrestre",
    "Eolien offshore",
]


DIM_COLUMNS = [
    "Périmètre",
    "Nature",
    "Date",
    "Heures",
    "datetime",
]

In [13]:
FLOW_COLUMNS = df.columns[df.columns.str.contains("Flux", regex=True)].tolist()

In [14]:
RATE_COLUMNS = [col for col in df.columns if col.startswith(("TCO", "TCH"))]


In [15]:
print("Dimensions :", DIM_COLUMNS)
print("Energy :", ENERGY_COLUMNS)
print("Flows :", FLOW_COLUMNS)
print("Rates :", RATE_COLUMNS)


Dimensions : ['Périmètre', 'Nature', 'Date', 'Heures', 'datetime']
Energy : ['Consommation', 'Thermique', 'Nucléaire', 'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies', 'Ech. physiques', 'Stockage batterie', 'Déstockage batterie', 'Eolien terrestre', 'Eolien offshore']
Flows : ['Flux physiques de Normandie vers Bretagne', 'Flux physiques de Pays-de-la-Loire vers Bretagne', 'Flux physiques de Bretagne vers Normandie', 'Flux physiques de Bretagne vers Pays-de-la-Loire']
Rates : ['TCO Thermique (%)', 'TCH Thermique (%)', 'TCO Nucléaire (%)', 'TCO Eolien (%)', 'TCH Eolien (%)', 'TCO Solaire (%)', 'TCH Solaire (%)', 'TCO Hydraulique (%)', 'TCH Hydraulique (%)', 'TCO Bioénergies (%)', 'TCH Bioénergies (%)']


In [16]:
analytics_columns = DIM_COLUMNS + ENERGY_COLUMNS
analytics_df = df[[col for col in analytics_columns if col in df.columns]].copy()
flows_df = df[["datetime", "Périmètre"] + FLOW_COLUMNS].copy()
rates_df = df[["datetime", "Périmètre"] + RATE_COLUMNS].copy()

filename ={
    "bretagne_analysis": analytics_df ,
    "bretagne_flows":flows_df ,
    "bretagne_rates_energy" :rates_df }

for file , data in filename.items() :
    data.to_parquet(f"data/staging/{file}.parquet" , index=False )# , engine ="pyarrow")

## Données de Auvergne-Rhône-Alpes

In [51]:
RAW_DATA = PROJECT_DIR / r"data\raw\Auvergne-Rhône-Alpes"
test = pd.read_csv(RAW_DATA / "eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2013.xls" ,
sep="\t" ,
encoding="latin1",
index_col= False ,
na_values= ['ND' ,"NaN" , "-"]
)

test['datatime'] = pd.to_datetime(test['Date'].astype(str)+ " " + test['Heures'].astype(str) ,
                                  format = "%Y-%m-%d %H:%M" ,
                                  errors ="coerce")

test.head(5)


,Périmètre,Nature,Date,Heures,Consommation,Thermique,Nucléaire,Eolien,Solaire,Hydraulique,Pompage,Bioénergies,Ech. physiques,Unnamed: 13,datatime
0,Auvergne-Rhône-Alpes,Données définitives,2013-01-01,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013-01-01 00:00:00
1,Auvergne-Rhône-Alpes,Données définitives,2013-01-01,00:15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013-01-01 00:15:00
2,Auvergne-Rhône-Alpes,Données définitives,2013-01-01,00:30,8173.0,252.0,12069.0,257.0,0.0,3079.0,-4.0,68.0,-7551.0,NaN,2013-01-01 00:30:00
3,Auvergne-Rhône-Alpes,Données définitives,2013-01-01,00:45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013-01-01 00:45:00
4,Auvergne-Rhône-Alpes,Données définitives,2013-01-01,01:00,7944.0,252.0,11895.0,257.0,0.0,2828.0,-328.0,67.0,-7029.0,NaN,2013-01-01 01:00:00


In [37]:
for col in test.columns :
    vals=test[col].value_counts(ascending=False)
    print(f"{col:<30} {vals.index[0:3]}")

Périmètre                      Index(['Auvergne-Rhône-Alpes', 'RTE ne pourra être tenu responsable de l'usage qui pourrait être fait des données mises à disposition, ni en cas de prévisions qui se révèleraient imprécises.'], dtype='str', name='Périmètre')
Nature                         Index(['Données définitives'], dtype='str', name='Nature')
Date                           Index(['2013-01-01', '2013-01-02', '2013-01-03'], dtype='str', name='Date')
Heures                         Index(['00:00', '00:15', '00:30'], dtype='str', name='Heures')
Consommation                   Index([5372.0, 7319.0, 6687.0], dtype='float64', name='Consommation')
Thermique                      Index([50.0, 41.0, 25.0], dtype='float64', name='Thermique')
Nucléaire                      Index([13492.0, 12622.0, 12626.0], dtype='float64', name='Nucléaire')
Eolien                         Index([52.0, 62.0, 50.0], dtype='float64', name='Eolien')
Solaire                        Index([0.0, 1.0, 2.0], dtype='float64',

In [84]:
reference_columns = set(all_data[0].columns)

for i, data in enumerate(all_data):
    current_columns = set(data.columns)

    missing = reference_columns - current_columns
    extra = current_columns - reference_columns

    print(f"\nDataFrame {all_data[i]['Date'].str[:4].iloc[0]}")

    if not missing and not extra:
        print("✅ Même variables")
    else:
        print("❌ Variables différentes")
        print("Colonnes manquantes :", missing)
        print("Colonnes supplémentaires :", extra)



DataFrame 2013
✅ Même variables

DataFrame 2014
✅ Même variables

DataFrame 2015
✅ Même variables

DataFrame 2016
✅ Même variables

DataFrame 2017
✅ Même variables

DataFrame 2018
✅ Même variables

DataFrame 2019
✅ Même variables

DataFrame 2020
❌ Variables différentes
Colonnes manquantes : set()
Colonnes supplémentaires : {'TCO Thermique (%)', 'TCH Hydraulique (%)', 'TCH Eolien (%)', 'TCO Hydraulique (%)', 'TCO Solaire (%)', 'TCH Bioénergies (%)', 'TCH Nucléaire (%)', 'TCO Nucléaire (%)', 'TCH Solaire (%)', 'TCO Eolien (%)', 'TCO Bioénergies (%)', 'TCH Thermique (%)'}

DataFrame 2021
❌ Variables différentes
Colonnes manquantes : set()
Colonnes supplémentaires : {'TCH Hydraulique (%)', 'Flux physiques de Nouvelle-Aquitaine vers Auvergne-Rhône-Alpes', 'TCH Solaire (%)', 'TCO Bioénergies (%)', 'Flux physiques de Auvergne-Rhône-Alpes vers Occitanie', "Flux physiques d'Occitanie vers Auvergne-Rhône-Alpes", 'Flux physiques de Auvergne-Rhône-Alpes vers Suisse', 'Flux physiques Italie vers A

In [48]:
test.isna().sum()

Périmètre             0
Nature                1
Date                  1
Heures                1
Consommation      17522
Thermique         17522
Nucléaire         17522
Eolien            17522
Solaire           17522
Hydraulique       17522
Pompage           17522
Bioénergies       17522
Ech. physiques    17522
Unnamed: 13       35041
datatime              1
dtype: int64

In [ ]:
all_data = []
 
for data in RAW_DATA.iterdir():
    auvg_df = pd.read_csv(
        data,
        sep="\t",
        encoding="latin1",
        dtype=str,
        na_values=["-", "", "NA", "N/A", "ND"],
        keep_default_na=True,
        index_col=False,
    )

    auvg_df = auvg_df.iloc[:-1].copy()

    auvg_df = auvg_df.loc[:, ~auvg_df.columns.str.startswith("Unnamed")].copy()

    auvg_df["datetime"] = pd.to_datetime(
        auvg_df["Date"] + " " + auvg_df["Heures"], errors="coerce", dayfirst=True
    )
    auvg_df.drop(columns = ['Nature' ,'Date' ,"Heures"])
    
    auvg_df.dropna(how="all" , axis=1 , inplace=True)
    auvg_df = auvg_df[auvg_df["datetime"].dt.minute.isin([0, 30])].copy()
    auvg_df = auvg_df.drop(index=0 , axis=0 , inplace=True)
    auvg_df_cols = auvg_df.columns[~auvg_df.columns.str.contains('Flux')]
    auvg_df= auvg_df[auvg_df_cols]

    NUMERIC_VARS = test.columns[
        ~test.columns.isin(
            {"datetime", "Périmètre", }
        )
    ]


for var in NUMERIC_VARS:
    auvg_df[var] = auvg_df[var].astype(float)

    all_data.append(auvg_df)


eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2013.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2014.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2015.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2016.xls | 17,568 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2017.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2018.xls | 17,520 lignes | 13 colonnes


C:\Users\user\AppData\Local\Temp\ipykernel_9744\1108976165.py:26: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2019.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2020.xls | 17,568 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2021.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2022.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2023.xls | 17,520 lignes | 13 colonnes
eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2024.xls | 17,568 lignes | 13 colonnes

Dataset final : (210384, 13)


In [119]:
# 1. Valeurs manquantes par colonne
missing = auvg_df.isna().mean().mul(100).round(2).sort_values(ascending=False)

print("Valeurs manquantes (%)")
print(missing)

# 2. Doublons temporels
duplicates = auvg_df.duplicated(subset=["Périmètre", "datetime"]).sum()

print(f"\nDoublons temporels : {duplicates}")

# 3. Nombre d'observations par année
annual_counts = auvg_df.groupby(auvg_df["datetime"].dt.year).size()

print("\nObservations par année")
print(annual_counts)

# 4. Vérification des types
print("\nTypes des colonnes")
print(auvg_df.dtypes)


Valeurs manquantes (%)
Eolien terrestre    66.67
Eolien offshore     66.67
Consommation         0.00
datetime             0.00
Périmètre            0.00
Thermique            0.00
Nucléaire            0.00
Eolien               0.00
Solaire              0.00
Pompage              0.00
Hydraulique          0.00
Ech. physiques       0.00
Bioénergies          0.00
dtype: float64

Doublons temporels : 0

Observations par année
datetime
2013    17520
2014    17520
2015    17520
2016    17568
2017    17520
2018    17520
2019    17520
2020    17568
2021    17520
2022    17520
2023    17520
2024    17568
dtype: int64

Types des colonnes
Périmètre                      str
datetime            datetime64[us]
Consommation               float64
Nucléaire                  float64
Thermique                  float64
Eolien                     float64
Solaire                    float64
Hydraulique                float64
Pompage                    float64
Bioénergies                float64
Ech. physiques  

In [ ]:
test.drop(index=0 , axis=0 )


,Périmètre,Nature,Date,Heures,Consommation,Thermique,Nucléaire,Eolien,Solaire,Hydraulique,Pompage,Bioénergies,Ech. physiques,TCO Thermique (%),TCH Thermique (%),TCO Nucléaire (%),TCH Nucléaire (%),TCO Eolien (%),TCH Eolien (%),TCO Solaire (%),TCH Solaire (%),TCO Hydraulique (%),TCH Hydraulique (%),TCO Bioénergies (%),TCH Bioénergies (%),datetime
2,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,00:30,8814,221,11555,38,0,3962,-4,144,-7103,2.51,20.89,131.1,85.15,0.43,5.82,0,0,44.95,35.67,1.63,52.94,2020-01-01 00:30:00
4,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,01:00,8589,221,11439,34,0,3840,-4,146,-7087,2.57,20.89,133.18,84.3,0.4,5.21,0,0,44.71,34.57,1.7,53.68,2020-01-01 01:00:00
6,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,01:30,8575,221,11618,38,0,3786,-4,147,-7231,2.58,20.89,135.49,85.62,0.44,5.82,0,0,44.15,34.09,1.71,54.04,2020-01-01 01:30:00
8,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,02:00,8560,223,11663,38,0,3382,-4,147,-6888,2.61,21.08,136.25,85.95,0.44,5.82,0,0,39.51,30.45,1.72,54.04,2020-01-01 02:00:00
10,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,02:30,8639,223,11666,38,0,3166,-4,146,-6596,2.58,21.08,135.04,85.97,0.44,5.82,0,0,36.65,28.5,1.69,53.68,2020-01-01 02:30:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33302,Auvergne-Rhône-Alpes,Données définitives,2020-12-12,21:30,7961,189,10653,211,0,2912,-329,127,-5803,2.37,17.71,133.81,78.5,2.65,37.75,0,0,36.58,24.69,1.6,68.28,2020-12-12 21:30:00
33304,Auvergne-Rhône-Alpes,Données définitives,2020-12-12,22:00,7806,187,10627,218,0,2473,-329,127,-5496,2.4,17.53,136.14,78.31,2.79,39,0,0,31.68,20.97,1.63,68.28,2020-12-12 22:00:00
33306,Auvergne-Rhône-Alpes,Données définitives,2020-12-12,22:30,7962,187,10683,222,0,2418,-329,128,-5348,2.35,17.53,134.17,78.73,2.79,39.71,0,0,30.37,20.5,1.61,68.82,2020-12-12 22:30:00
33308,Auvergne-Rhône-Alpes,Données définitives,2020-12-12,23:00,8406,187,10662,194,0,2957,-367,129,-5356,2.22,17.53,126.84,78.57,2.31,34.7,0,0,35.18,25.07,1.53,69.35,2020-12-12 23:00:00


In [96]:
for file_path in RAW_DATA.iterdir() :
    print(file_path)
    with open(file_path, encoding="latin1") as f:
        lines = f.readlines()

    print("En-tête :", repr(lines[0]))
    print("Première ligne :", repr(lines[1]))
    print("Dernière ligne :", repr(lines[-1]))


C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte\data\raw\Auvergne-Rhône-Alpes\eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2013.xls
En-tête : 'Périmètre\tNature\tDate\tHeures\tConsommation\tThermique\tNucléaire\tEolien\tSolaire\tHydraulique\tPompage\tBioénergies\tEch. physiques\t\n'
Première ligne : 'Auvergne-Rhône-Alpes\tDonnées définitives\t2013-01-01\t00:00\tND\tND\tND\tND\tND\tND\tND\tND\tND\t\n'
Dernière ligne : '\n'
C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte\data\raw\Auvergne-Rhône-Alpes\eCO2mix_RTE_Auvergne-Rhône-Alpes_Annuel-Definitif_2014.xls
En-tête : 'Périmètre\tNature\tDate\tHeures\tConsommation\tThermique\tNucléaire\tEolien\tSolaire\tHydraulique\tPompage\tBioénergies\tEch. physiques\t\n'
Première ligne : 'Auvergne-Rhône-Alpes\tDonnées définitives\t2014-01-01\t00:00\t8653\t189\t13192\t130\t0\t3868\t-6\t77\t-8799\t\n'
Dernière ligne : '\n'
C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte\data\raw\Auvergne-Rhône-Alpes\eCO2mix_RTE_A

In [102]:
auvg_df['Périmètre'].astype(str)[0]


'Auvergne-Rhône-Alpes'

In [ ]:
for auvg_df in all_data :
    print(f"{auvg_df['Périmètre'].astype(str)[0] + ""+auvg_df['Date'].astype(str)[0][:4]}")
    print("Dimensions :", auvg_df.shape)

    print("\nValeurs manquantes :")
    print(auvg_df.isna().sum())

    print("\nDoublons :")
    print(auvg_df.duplicated(subset=["Périmètre", "Date", "Heures"]).sum())


Auvergne-Rhône-Alpes2013
Dimensions : (6912, 14)

Valeurs manquantes :
Périmètre         0
Nature            0
Date              0
Heures            0
Consommation      1
Thermique         1
Nucléaire         1
Eolien            1
Solaire           1
Hydraulique       1
Pompage           1
Bioénergies       1
Ech. physiques    1
datetime          0
dtype: int64

Doublons :
0
Auvergne-Rhône-Alpes2014
Dimensions : (6912, 14)

Valeurs manquantes :
Périmètre         0
Nature            0
Date              0
Heures            0
Consommation      0
Thermique         0
Nucléaire         0
Eolien            0
Solaire           0
Hydraulique       0
Pompage           0
Bioénergies       0
Ech. physiques    0
datetime          0
dtype: int64

Doublons :
0
Auvergne-Rhône-Alpes2015
Dimensions : (6912, 14)

Valeurs manquantes :
Périmètre         0
Nature            0
Date              0
Heures            0
Consommation      0
Thermique         0
Nucléaire         0
Eolien            0
Solaire      

In [83]:
for  i , data in enumerate(all_data) :
    print(f"Fichier {i+1} presente :{data.shape[0]} observations , {data.shape[1]} variables et voici les variables {data.columns.tolist()}")

Fichier 1 presente :6912 observations , 14 variables et voici les variables ['Périmètre', 'Nature', 'Date', 'Heures', 'Consommation', 'Thermique', 'Nucléaire', 'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies', 'Ech. physiques', 'datetime']
Fichier 2 presente :6912 observations , 14 variables et voici les variables ['Périmètre', 'Nature', 'Date', 'Heures', 'Consommation', 'Thermique', 'Nucléaire', 'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies', 'Ech. physiques', 'datetime']
Fichier 3 presente :6912 observations , 14 variables et voici les variables ['Périmètre', 'Nature', 'Date', 'Heures', 'Consommation', 'Thermique', 'Nucléaire', 'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies', 'Ech. physiques', 'datetime']
Fichier 4 presente :6912 observations , 14 variables et voici les variables ['Périmètre', 'Nature', 'Date', 'Heures', 'Consommation', 'Thermique', 'Nucléaire', 'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies', 'Ech. physiques', 'dat

In [55]:
COMMON_COLUMNS_ANALYSIS  = [
    "Périmètre",
    "datetime",
    "Consommation",
    "Thermique",
    "Eolien",
    "Solaire",
    "Hydraulique",
    "Pompage",
    "Bioénergies",
    "Ech. physiques",
    "Eolien terrestre",
    "Eolien offshore",
]

In [106]:
test = all_data[7]

In [107]:
test.head(5)

,Périmètre,Nature,Date,Heures,Consommation,Thermique,Nucléaire,Eolien,Solaire,Hydraulique,Pompage,Bioénergies,Ech. physiques,TCO Thermique (%),TCH Thermique (%),TCO Nucléaire (%),TCH Nucléaire (%),TCO Eolien (%),TCH Eolien (%),TCO Solaire (%),TCH Solaire (%),TCO Hydraulique (%),TCH Hydraulique (%),TCO Bioénergies (%),TCH Bioénergies (%),datetime
0,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,00:00,8911,220,11556,41,0,3663,-8,144,-6706,2.47,20.79,129.68,85.16,0.46,6.28,0,0,41.11,32.98,1.62,52.94,2020-01-01 00:00:00
2,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,00:30,8814,221,11555,38,0,3962,-4,144,-7103,2.51,20.89,131.1,85.15,0.43,5.82,0,0,44.95,35.67,1.63,52.94,2020-01-01 00:30:00
4,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,01:00,8589,221,11439,34,0,3840,-4,146,-7087,2.57,20.89,133.18,84.3,0.4,5.21,0,0,44.71,34.57,1.7,53.68,2020-01-01 01:00:00
6,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,01:30,8575,221,11618,38,0,3786,-4,147,-7231,2.58,20.89,135.49,85.62,0.44,5.82,0,0,44.15,34.09,1.71,54.04,2020-01-01 01:30:00
8,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,02:00,8560,223,11663,38,0,3382,-4,147,-6888,2.61,21.08,136.25,85.95,0.44,5.82,0,0,39.51,30.45,1.72,54.04,2020-01-01 02:00:00


In [64]:
len(test)

6912

In [ ]:
# test.isna().sum() == len(test)
# test.dropna(how="all" , axis=1 , inplace=True)

,Périmètre,Nature,Date,Heures,Consommation,Thermique,Nucléaire,Eolien,Solaire,Hydraulique,Pompage,Bioénergies,Ech. physiques,TCO Thermique (%),TCH Thermique (%),TCO Nucléaire (%),TCH Nucléaire (%),TCO Eolien (%),TCH Eolien (%),TCO Solaire (%),TCH Solaire (%),TCO Hydraulique (%),TCH Hydraulique (%),TCO Bioénergies (%),TCH Bioénergies (%),datetime
0,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,00:00,8911.0,220.0,11556.0,41.0,0.0,3663.0,-8.0,144.0,-6706.0,2.47,20.79,129.68,85.16,0.46,6.28,0.0,0.0,41.11,32.98,1.62,52.94,2020-01-01 00:00:00
2,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,00:30,8814.0,221.0,11555.0,38.0,0.0,3962.0,-4.0,144.0,-7103.0,2.51,20.89,131.10,85.15,0.43,5.82,0.0,0.0,44.95,35.67,1.63,52.94,2020-01-01 00:30:00
4,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,01:00,8589.0,221.0,11439.0,34.0,0.0,3840.0,-4.0,146.0,-7087.0,2.57,20.89,133.18,84.30,0.40,5.21,0.0,0.0,44.71,34.57,1.70,53.68,2020-01-01 01:00:00
6,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,01:30,8575.0,221.0,11618.0,38.0,0.0,3786.0,-4.0,147.0,-7231.0,2.58,20.89,135.49,85.62,0.44,5.82,0.0,0.0,44.15,34.09,1.71,54.04,2020-01-01 01:30:00
8,Auvergne-Rhône-Alpes,Données définitives,2020-01-01,02:00,8560.0,223.0,11663.0,38.0,0.0,3382.0,-4.0,147.0,-6888.0,2.61,21.08,136.25,85.95,0.44,5.82,0.0,0.0,39.51,30.45,1.72,54.04,2020-01-01 02:00:00


In [87]:
test_1 = all_data[-3]
test_1.isna().sum()

Périmètre                                                                 0
Nature                                                                    0
Date                                                                      0
Heures                                                                    0
Consommation                                                              0
Thermique                                                                 0
Nucléaire                                                                 0
Eolien                                                                    0
Solaire                                                                   0
Hydraulique                                                               0
Pompage                                                                   0
Bioénergies                                                               0
Stockage batterie                                                         0
Déstockage b